# ML-10 — Content Action Playbook

This notebook turns the validated queue into a human-reviewed worklist. It is decision support only; it does not predict Google’s algorithm or automate content changes.

## 1. Ranked actions + reason codes

Ranked rows are review candidates. Reason codes explain why a human should inspect an item; they are not automatic instructions.

In [1]:
from pathlib import Path
import csv
rows = [
 {'final_rank':1,'content_id':'content_1f080331fa2b','score':81.64,'confidence':'high','action':'refresh_and_review_ctr','reason_codes':'declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity'},
 {'final_rank':2,'content_id':'content_6aa43079fb0c','score':81.45,'confidence':'high','action':'refresh_and_review_ctr','reason_codes':'declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity'},
 {'final_rank':3,'content_id':'content_d6570c51c9bd','score':81.43,'confidence':'medium','action':'refresh_and_review_ctr','reason_codes':'declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity'},
 {'final_rank':4,'content_id':'content_72e800a9c214','score':81.03,'confidence':'high','action':'refresh_and_review_ctr','reason_codes':'declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity'},
 {'final_rank':5,'content_id':'content_e04eb9549989','score':80.87,'confidence':'medium','action':'refresh_and_review_ctr','reason_codes':'declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity'}]
for row in rows:
    row['review_step'] = 'Inspect intent, evidence, title/snippet, and usefulness before proposing a change.'
    row['decision'] = 'human_review_required'
out = Path('work/outputs'); out.mkdir(parents=True, exist_ok=True)
with (out/'w07_ranked_action_queue.csv').open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader(); writer.writerows(rows)
print('Exported', len(rows), 'ranked review items to', out/'w07_ranked_action_queue.csv')


Exported 5 ranked review items to work/outputs/w07_ranked_action_queue.csv


### Archetype → action mapping

| Observed archetype | Reason codes | Human action |
|---|---|---|
| Declining with visible demand | declining_with_demand, visible_model_opportunity | Inspect current intent match and whether a refresh is warranted. |
| Visible but low CTR | low_ctr_visible_page | Compare title/snippet with current results; propose and review a test. |
| Visible but low engagement | low_engagement_visible_page | Check usefulness, structure, speed, and internal links. |
| Older visible content | page_one_decay_risk | Check facts, examples, and competing pages for staleness. |

Decay/refresh insight: visible older items may deserve a freshness check, but this observational dataset does not show that age caused a decline or that refreshing will improve an outcome.

## 2. Intended use and limits

A content lead uses this queue during weekly planning to choose pseudonymized items to inspect first. Scores combine existing baseline/model outputs and are traceable through reason codes. This is a trailing-90-day observational snapshot: it cannot establish that any edit will increase traffic. Scores can drift across clients, content types, and time. Low-confidence items are review leads, not commitments.

## 3. Human review + no-go list

Before acting, a reviewer confirms that the item is live and relevant, the reason codes match observed evidence, the proposed change is accurate and brand-safe, and the effort is justified.

Never automate: publishing or rewriting content; title/meta/internal-link changes; deletion or redirects; health/legal/financial/regulatory claims; client contact; or prioritization without human review.

## 4. Monitoring / retrain triggers

Monitor reason-code mix, score distribution by client/content type, missingness, reviewer acceptance/rejection, and held-out outcome trends. Pause and investigate if feature distributions or missingness shift, top-queue acceptance falls, unvalidated content types dominate, or scores become implausible. Retrain only after checking label definition, leakage controls, grouped/time-aware validation, and enough newly reviewed outcomes; compare any replacement against the current model on the same held-out design.

## 5. Cost/value thinking and exports

A high score is not automatically high value. Estimate review/edit cost, opportunity size, and downside before work begins. Start with high-confidence visible items whose review cost is low; defer expensive rewrites until a reviewer identifies a concrete issue. The code exports work/outputs/w07_ranked_action_queue.csv for the paper workflow; it remains out of git by design.

## Self-check

- [x] Ranked actions, reason codes, and archetype map
- [x] Intended use, limits, human review, no-go list
- [x] Monitoring/retrain and cost/value rules
- [x] Generated queue export
- [x] Observational, decision-support claims only